# Incident Triage Assistant

When a website or app breaks, an engineer has to work out **what broke and why**. This project uses Claude as a helper for that.

1. **You describe the problem** in normal words, for example: *"Checkout is broken since 2pm."*
2. **Claude checks the facts** by asking your Python code for information, such as which services exist, how many errors each one has, and whether anything was changed recently. (The data is pretend, just for practice.)
3. **Claude works out the most likely cause**, for example: *"A new update at 1:52pm probably broke checkout."*
4. **Claude writes a short report in a fixed format:** how serious it is, what's affected, the likely cause, the evidence, what to do next, and how confident it is.
5. **Python checks the report is correct and complete.** If something is wrong, such as a service that doesn't exist, Python tells Claude, and Claude fixes it.

**What you'll learn:** how to call Claude from Python, how to let Claude use your own functions (tools), and how to get answers in a fixed format that your code can check.

---

In [ ]:
# Step 1. Send a question to Claude and display its answer in Markdown.

from anthropic import Anthropic  # Import the class used to connect to Claude's API.
from dotenv import load_dotenv, find_dotenv  # Import functions to load and locate a .env file.

load_dotenv(find_dotenv())  # Find the .env file and load its settings, including your API key.
client = Anthropic()  # client stores the API connection, using the loaded key.
MODEL = "claude-haiku-4-5"  # MODEL stores the name of the Claude model to use.

response = client.messages.create(  # Send a request; response stores Claude's complete reply.
    model=MODEL,  # Select the model named in MODEL.
    max_tokens=200,  # Limit the generated response to 200 tokens.
    messages=[{"role": "user", "content": "In one sentence, what is an on-call engineer?"}],  # Send a list containing one user question.
)  # Finish the API request.

from IPython.display import display, Markdown  # Import functions for displaying formatted Markdown in Jupyter.

display(Markdown(response.content[0].text))  # Extract the first response block's text and display it as Markdown.

An on-call engineer is a software engineer who is available to respond to and resolve urgent production issues or incidents outside of regular business hours.

## **Step 2: Create the fake service data**

- **`checkout-service`:** where customers place their order. **Broken** (many errors, slow), and it was **updated at 13:52**, just before the problem.
- **`payments-api`:** charges the customer's card. **Healthy.**
- **`auth-service`:** handles logging in. **Healthy.** Only a small update this morning.

**Terms:**
- **Error rate:** the percentage of requests that failed.
- **p95 latency:** how slow the service is, in milliseconds.
- **Deploy:** releasing a new version of the code.

In [ ]:
# 2. Store example service monitoring data and read specific values.

SERVICES = {  # SERVICES is a dictionary mapping service names to their details.
    "checkout-service": {  # Store details for the checkout service.
        "error_rate_pct": 7.4,  # Percentage of requests that fail: 7.4%.
        "p95_latency_ms": 2300,  # 95% of requests take at most 2,300 milliseconds.
        "recent_deploys": [  # List recent software releases for this service.
            {  # Store details of one release.
                "version": "v2.18.0",  # version identifies the software release.
                "deployed_at": "13:52",  # deployed_at records the release time as text.
                "change": "New promo-code discount logic",  # change describes what was updated.
            },  # Finish this release's details.
        ],  # Finish the list of releases.
    },  # Finish the checkout service's details.
    "payments-api": {  # Store details for the payments service.
        "error_rate_pct": 0.3,  # Percentage of requests that fail: 0.3%.
        "p95_latency_ms": 410,  # 95% of requests take at most 410 milliseconds.
        "recent_deploys": [],  # An empty list means no recent releases are recorded.
    },  # Finish the payments service's details.
    "auth-service": {  # Store details for the authentication service.
        "error_rate_pct": 0.1,  # Percentage of requests that fail: 0.1%.
        "p95_latency_ms": 120,  # 95% of requests take at most 120 milliseconds.
        "recent_deploys": [  # List recent software releases for this service.
            {  # Store details of one release.
                "version": "v5.2.1",  # version identifies the software release.
                "deployed_at": "09:10",  # deployed_at records the release time as text.
                "change": "Dependency bump",  # A software library or package was updated.
            },  # Finish this release's details.
        ],  # Finish the list of releases.
    },  # Finish the authentication service's details.
}  # Finish the SERVICES dictionary.

In [9]:
print(list(SERVICES))  # Print the dictionary's keys: the three service names.
print(SERVICES["checkout-service"]["error_rate_pct"])  # Read and print checkout's error percentage: 7.4.
print(SERVICES["payments-api"]["recent_deploys"])  # Read and print payments' release list: [].

['checkout-service', 'payments-api', 'auth-service']
7.4
[]


## **Step 3: write the local tool functions**

**Goal**: Write the two Python functions that Claude will be allowed to ***call***. They read from your `SERVICES` dictionary.

In [12]:
# 3. List available services and retrieve their monitoring details.

def list_services():  # Define a function that takes no arguments.
    """Return service names in alphabetical order."""  # Describe the function.
    return sorted(SERVICES)  # Return a sorted list of keys from the SERVICES dictionary.


def get_service_metrics(service_name):  # service_name is the name of the service to look up.
    """Return the service's name and monitoring details."""  # Describe the function.
    if service_name not in SERVICES:  # Check whether the requested name is missing from SERVICES.
        raise ValueError(f"Unknown service '{service_name}'. Call list_services first.")  # Stop this call with an error showing the unknown name.
    return {"service": service_name, **SERVICES[service_name]}  # Create a dictionary with the name; ** copies in the service's stored details.

In [13]:
print(list_services())  # Call the function and print the alphabetically sorted service names.
# Output: ['auth-service', 'checkout-service', 'payments-api']

print(get_service_metrics("payments-api"))  # Retrieve and print the payments service's details.
# Output: {'service': 'payments-api', 'error_rate_pct': 0.3, 'p95_latency_ms': 410, 'recent_deploys': []}

get_service_metrics("billing")  # Request an unknown service to demonstrate error handling.
# Raises ValueError: Unknown service 'billing'. Call list_services first.

['auth-service', 'checkout-service', 'payments-api']
{'service': 'payments-api', 'error_rate_pct': 0.3, 'p95_latency_ms': 410, 'recent_deploys': []}


ValueError: Unknown service 'billing'. Call list_services first.

## **Step 4: Write the tool schemas**

**Goal**: Describe your two functions to Claude. A *schema* tells Claude the tool’s name, what it does and what inputs it needs, so it knows when and how to ask for it. Claude only sees the schemas, never your Python code.

Make a list called `TOOLS` with two schema dictionaries, one per function.

In [85]:
# 4. Describe the available tools and let Claude request one.

TOOLS = [  # TOOLS stores a list of tool descriptions; it does not execute the functions.
    {  # Start the description of the first tool.
        "name": "list_services",  # Name of the function Claude can request.
        "description": "Lists all services in production. Use this before looking up metrics.",  # Explain its purpose and when to use it.
        "input_schema": {  # Describe the arguments this tool accepts.
            "type": "object",  # Arguments must be supplied as a JSON object.
            "properties": {},  # This tool has no defined input arguments.
            "required": [],  # No arguments are required.
        },  # Finish the input schema.
    },  # Finish the first tool's description.
    {  # Start the description of the second tool.
        "name": "get_service_metrics",  # Name of the function Claude can request.
        "description": "Returns the current error rate (%), p95 latency (ms) and recent deploys for one service. Use it to check a service's health during an incident.",  # Explain its purpose.
        "input_schema": {  # Describe the arguments this tool accepts.
            "type": "object",  # Arguments must be supplied as a JSON object.
            "properties": {  # List the available input arguments.
                "service_name": {  # service_name identifies the service to inspect.
                    "type": "string",  # The service name must be text.
                    "description": "Exact service name",  # Tell Claude to supply the precise name.
                },  # Finish describing service_name.
            },  # Finish listing the arguments.
            "required": ["service_name"],  # Require Claude to provide service_name.
        },  # Finish the input schema.
    },  # Finish the second tool's description.
]  # Finish the TOOLS list.

In [86]:
print([tool["name"] for tool in TOOLS])  # Loop through TOOLS; tool holds each dictionary, and ["name"] extracts its name.
# Output: ['list_services', 'get_service_metrics']

['list_services', 'get_service_metrics']


In [87]:
response = client.messages.create(  # Send a request; response stores Claude's complete reply.
    model=MODEL,  # Use the model name stored in MODEL.
    max_tokens=300,  # Limit the generated response to 300 tokens.
    tools=TOOLS,  # Tell Claude which tools it can request.
    messages=[  # Supply the conversation as a list.
        {  # Start one user message.
            "role": "user",  # Identify the message as coming from the user.
            "content": "Which services do we have?",  # Supply the user's question.
        },  # Finish the user message.
    ],  # Finish the conversation list.
)  # Finish the API request.

print(response.stop_reason)  # Print why Claude stopped; expect "tool_use" if it requests a tool.
print(response.content)  # Print the response blocks; look for a tool request named "list_services".

tool_use
[ToolUseBlock(id='toolu_01JLKXMTMd9JcDdrqgnJjLnx', caller=DirectCaller(type='direct'), input={}, name='list_services', type='tool_use', toolset_name=None)]


## **Step 5: the router and `run_tools()`**

**Goal:** when Claude asks for a tool, your code must **run the right Python function** and package the result so Claude can read it. You'll write two things:

| Part | Job |
|---|---|
| **`LOCAL_TOOLS`** (the router) | A dictionary linking each **tool name** to its **Python function** |
| **`run_tools(response)`** | Goes through Claude's reply, runs **every** tool requested and returns a list of **tool result** blocks |

In [88]:
# 5. Execute Claude's requested tools and collect their results or errors.

import json  # Import utilities for converting Python values into JSON text.

LOCAL_TOOLS = {  # LOCAL_TOOLS maps tool names to the actual Python functions.
    "list_services": list_services,  # Store the list_services function without calling it.
    "get_service_metrics": get_service_metrics,  # Store the get_service_metrics function without calling it.
}  # Finish the function lookup dictionary.


def run_tools(response):  # response contains Claude's complete reply, including tool requests.
    results = []  # results will store the result of each requested tool.

    for block in response.content:  # block holds each part of Claude's reply.
        if block.type != "tool_use":  # Check whether this block is something other than a tool request.
            continue  # Skip this block and move to the next one.
        print(f"-> Claude called {block.name}({block.input})")   # NEW: show each tool request
        try:  # Attempt to run the tool and prepare its result.
            output = LOCAL_TOOLS[block.name](**block.input)  # Find the function by name, unpack its arguments and store its return value in output.
            results.append({  # Add a successful result dictionary to results.
                "type": "tool_result",  # Identify this block as a tool result.
                "tool_use_id": block.id,  # Match this result to the original tool request.
                "content": json.dumps(output),  # Convert output into JSON text.
                "is_error": False,  # Indicate that the tool completed successfully.
            })  # Finish adding the successful result.

        except Exception as e:  # Catch an error and store it in e.
            results.append({  # Add an error-result dictionary to results.
                "type": "tool_result",  # Identify this block as a tool result.
                "tool_use_id": block.id,  # Match this error to the original tool request.
                "content": f"Error: {e}",  # Include the error's message as text.
                "is_error": True,  # Indicate that the tool failed.
            })  # Finish adding the error result.

    return results  # Return all collected tool results.

In [89]:
print(run_tools(response))  # Execute tools requested in response and print their results.
# Example: [{'type': 'tool_result', 'tool_use_id': 'toolu_...', 'content': '["auth-service", "checkout-service", "payments-api"]', 'is_error': False}]

-> Claude called list_services({})
[{'type': 'tool_result', 'tool_use_id': 'toolu_01JLKXMTMd9JcDdrqgnJjLnx', 'content': '["auth-service", "checkout-service", "payments-api"]', 'is_error': False}]


## **Step 6: the tool-use loop**

**Goal:** keep repeating **"ask Claude, run its tools, send the results back"** until Claude stops asking for tools and gives its final answer. You'll write:

| Part | Job |
|---|---|
| **`SYSTEM_PROMPT`** | Tells Claude its role and how to behave |
| **`chat(messages)`** | Sends the conversation to Claude **with the system prompt and your tools** |
| **`triage(incident)`** | The loop. It starts the conversation, repeats until Claude is done, and returns the final text |

**Triage** means investigating a problem to understand what is wrong and what needs attention first.

**Two details:** eight turns means eight Claude requests, not necessarily eight tool calls. Also, a stop reason other than `"tool_use"` can include reaching the token limit, so this code could return an incomplete answer.

In [104]:
# 6. Investigate an incident using tools, allowing up to eight Claude requests.

MAX_TURNS = 8  # MAX_TURNS limits how many times the loop can request a response from Claude.
SYSTEM_PROMPT = """You are an on-call incident triage assistant for an online shop.

How to investigate:
- Always call list_services first, then get_service_metrics for every service before concluding.
- Only cite facts returned by the tools. Never invent numbers, times or version numbers.
  Only mention a version if a tool returned it (for example, do not suggest rolling back to a version you were not given).

Match the report to the evidence:
- Identify which service the user's symptom relates to:
  checkout/basket/order -> checkout-service, card/payment -> payments-api, login/sign-in -> auth-service.
- Check that service's metrics. If they are healthy, say so clearly.
- Do NOT blame a different service just because it is unhealthy, unless the symptom clearly relates to it.
- Describe each service's health accurately (do not call a broken service "normal", or a healthy one "elevated").

Severity rules:
- SEV1: many customers cannot complete a core action (checkout, payment or login) AND the metrics show a major problem
  (error rate above 5% or p95 latency above 2000ms).
- SEV2: some customers are affected AND the metrics show a real, measurable degradation
  (error rate above 1% or p95 latency above 1000ms).
- SEV3: only one or a few reports, OR the related service's metrics look healthy.
- If unsure between two levels, choose the lower one unless the metrics clearly show widespread impact.

Finishing:
- Set confidence lower when the evidence is weak or the metrics do not support the report.
- When you are done, call submit_triage_report exactly once."""  # SYSTEM_PROMPT stores instructions for Claude.


def chat(messages):  # messages contains the conversation history.
    return client.messages.create(  # Send an API request and return Claude's complete response.
        model=MODEL,  # Use the model name stored in MODEL.
        max_tokens=1500,  # Limit this response to 1,500 generated tokens.
        system=SYSTEM_PROMPT,  # Supply the incident-investigation instructions.
        tools=TOOLS,  # Describe the tools Claude can request.
        messages=messages,  # Include the conversation and previous tool results.
    )  # Finish the API request.


def triage(incident):  # incident contains the user's description of the problem.
    messages = [{"role": "user", "content": incident}]  # Start the conversation with that description.

    for turn in range(MAX_TURNS):  # Repeat up to eight times; turn takes values from 0 to 7.
        response = chat(messages)  # Request Claude's next reply and store it in response.
        messages.append({"role": "assistant", "content": response.content})  # Save all reply blocks, including tool requests.

        if response.stop_reason != "tool_use":  # Check whether Claude stopped without requesting tool execution.
            return "\n".join(  # Combine text blocks with newlines and return the answer.
                block.text  # Take the text from this block.
                for block in response.content  # Examine each block in Claude's reply.
                if block.type == "text"  # Include only text blocks.
            )  # Finish combining the text; return also exits triage().

        messages.append({  # Add the tool results to the conversation.
            "role": "user",  # Tool results are sent back using the user role.
            "content": run_tools(response),  # Execute Claude's requested tools and store their result blocks.
        })  # Finish adding the results for the next loop iteration.

    raise RuntimeError(f"No answer after {MAX_TURNS} turns")  # Report failure if all eight iterations finish without returning an answer.

In [105]:
print(triage("Customers say checkout is failing with 500 errors since about 2pm."))  # Investigate the reported incident and print the returned answer.

-> Claude called list_services({})


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.2.content.0: Input should be an object'}, 'request_id': 'req_011Cg6CZ3sRUDZ9r97JVwasE'}

## **Step 7: the Pydantic model (structured, validated output)**

**Goal:** define **exactly what a triage report must look like**, using **Pydantic**, then turn it into a **tool** Claude must use to submit its answer. One model gives you two things:
1. **The JSON schema** for Claude, i.e. what fields to fill in
2. **Validation** in Python, to check Claude's answer follows the rules

**The report's fields and rules:**
| Field | Type | Rule |
|---|---|---|
| `severity` | text | Must be **exactly** `"SEV1"`, `"SEV2"` or `"SEV3"` |
| `summary` | text | At most **300 characters** |
| `affected_services` | list of text | At least **1** item, and each must be a **real service** |
| `likely_cause` | text | none |
| `evidence` | list of text | At least **1** item |
| `next_steps` | list of text | **1 to 5** items |
| `confidence` | number | Between **0 and 1** |

In [ ]:
# 7. Define and validate a structured incident report, then describe it as a tool for Claude.

from typing import List, Literal  # List describes lists; Literal restricts values to specific choices.
from pydantic import BaseModel, Field, ValidationError, field_validator  # Import tools for defining data rules, checking data and handling errors.


class TriageReport(BaseModel):  # TriageReport defines the required structure and validation rules for a report.
    severity: Literal["SEV1", "SEV2", "SEV3"] = Field(  # severity must be one of these three strings.
        description="SEV1 = major customer impact, SEV2 = partial, SEV3 = minor"  # Explain the severity levels.
    )  # Finish defining severity.

    summary: str = Field(  # summary contains a brief description as text.
        max_length=300,  # Allow at most 300 characters.
        description="One or two sentences describing the incident"  # Describe the intended content.
    )  # Finish defining summary.

    affected_services: List[str] = Field(  # affected_services contains a list of service names.
        min_length=1,  # Require at least one service name.
        description="Services that are degraded"  # Explain what belongs in the list.
    )  # Finish defining affected_services.

    likely_cause: str = Field(  # likely_cause contains the suspected cause as text.
        description="The most likely root cause, based on the evidence"  # Describe the expected content.
    )  # Finish defining likely_cause.

    evidence: List[str] = Field(  # evidence contains a list of supporting statements.
        min_length=1,  # Require at least one statement.
        description="Concrete facts from tool results"  # Describe the expected evidence.
    )  # Finish defining evidence.

    next_steps: List[str] = Field(  # next_steps contains a list of recommended actions.
        min_length=1,  # Require at least one action.
        max_length=5,  # Allow no more than five actions.
        description="Ordered actions for responders"  # Explain that actions should appear in their intended order.
    )  # Finish defining next_steps.

    confidence: float = Field(  # confidence contains a numerical confidence score.
        ge=0,  # ge means greater than or equal to zero.
        le=1,  # le means less than or equal to one.
        description="Confidence in likely_cause, from 0 to 1"  # Explain the score.
    )  # Finish defining confidence.

    @field_validator("affected_services")  # Run the following custom check on affected_services.
    @classmethod  # Make this method receive the class as its first argument.
    def services_must_exist(cls, value):  # cls refers to TriageReport; value is the service-name list being checked.
        unknown = [s for s in value if s not in SERVICES]  # unknown collects names missing from SERVICES; s is each name.
        if unknown:  # Check whether any unrecognised names were found.
            raise ValueError(f"Unknown services: {unknown}. Valid: {sorted(SERVICES)}")  # Report unknown names and list valid names alphabetically.
        return value  # Return the service list when all names are recognised.


TOOLS = [tool for tool in TOOLS if tool["name"] != "submit_triage_report"]  # remove any old copy
TOOLS.append({
    "name": "submit_triage_report",
    "description": "Submit the FINAL structured triage report. Call this once you have gathered evidence.",
    "input_schema": TriageReport.model_json_schema(),
})

In [106]:
good = {  # good stores example data intended to pass validation.
    "severity": "SEV1",  # Supply an allowed severity value.
    "summary": "Checkout is failing after a deploy.",  # Supply a short incident description.
    "affected_services": ["checkout-service"],  # Supply a service name present in SERVICES.
    "likely_cause": "v2.18.0 promo-code logic",  # Supply an example suspected cause.
    "evidence": ["Error rate 7.4%"],  # Supply one supporting statement.
    "next_steps": ["Roll back v2.18.0"],  # Supply one proposed action; this does not execute it.
    "confidence": 0.9,  # Supply a score within the allowed range.
}  # Finish the example report.

print(TriageReport.model_validate(good))  # Validate good, create a TriageReport object and print it.


bad = {  # bad stores a modified example intended to fail validation.
    **good,  # Copy all entries from good into this new dictionary.
    "severity": "SEV9",  # Replace severity with a value that is not allowed.
    "affected_services": ["billing"],  # Replace the service list with an unknown service.
    "confidence": 1.5,  # Replace confidence with a number above the maximum.
}  # Finish the invalid example.

try:  # Attempt to validate the invalid example.
    TriageReport.model_validate(bad)  # Check bad against the report's rules.
except ValidationError as e:  # Catch Pydantic's validation error and store it in e.
    print(e)  # Display the three validation errors.

severity='SEV1' summary='Checkout is failing after a deploy.' affected_services=['checkout-service'] likely_cause='v2.18.0 promo-code logic' evidence=['Error rate 7.4%'] next_steps=['Roll back v2.18.0'] confidence=0.9
3 validation errors for TriageReport
severity
  Input should be 'SEV1', 'SEV2' or 'SEV3' [type=literal_error, input_value='SEV9', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error
affected_services
  Value error, Unknown services: ['billing']. Valid: ['auth-service', 'checkout-service', 'payments-api'] [type=value_error, input_value=['billing'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.5, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


In [107]:
print([tool["name"] for tool in TOOLS])  # Extract and print each tool's name; tool holds each tool-description dictionary.
# Expected: ['list_services', 'get_service_metrics', 'submit_triage_report']

['list_services', 'get_service_metrics', 'submit_triage_report']


### Step 8: the repair loop and finishing touches

**Goal:** make Claude finish by **submitting a report**, **validate** it with your `TriageReport` model, and if it's invalid, **send the error back** so Claude fixes it. Then print the final, clean report.

In [108]:
# 1. Run tools, validate Claude's incident report and request corrections when validation fails.

def run_tools(response):  # response contains Claude's complete reply.
    results, report = [], None  # results stores tool replies; report stays None until a report passes validation.

    for block in response.content:  # block holds each part of Claude's reply.
        if block.type != "tool_use":  # Check whether this block is not a tool request.
            continue  # Skip it and examine the next block.

        print(f"-> Claude called {block.name}({block.input})")  # Display the requested tool's name and arguments.

        try:  # Attempt to process the tool request.
            if block.name == "submit_triage_report":  # Check whether Claude is submitting its final report.
                report = TriageReport.model_validate(block.input)  # Validate the arguments and store the resulting TriageReport object.
                output = "Report accepted."  # output stores the success message.
            else:  # Handle other tools, such as fetching service metrics.
                output = LOCAL_TOOLS[block.name](**block.input)  # Find the function, unpack its arguments and store its result.

            results.append({  # Add a successful tool reply to results.
                "type": "tool_result",  # Identify this entry as a tool result.
                "tool_use_id": block.id,  # Match the result to Claude's original request.
                "content": json.dumps(output),  # Convert output into JSON text.
                "is_error": False,  # Indicate that the request succeeded.
            })  # Finish adding the successful result.

        except ValidationError as e:  # Catch a Pydantic validation error and store it in e.
            print("   report failed validation -> asking Claude to fix it")  # Display a progress message.
            results.append({  # Add feedback that Claude can use to correct its report.
                "type": "tool_result",  # Identify this entry as a tool result.
                "tool_use_id": block.id,  # Match the error to the original request.
                "content": f"Validation failed, fix and resubmit:\n{e}",  # Include the validation errors after a newline.
                "is_error": True,  # Indicate that validation failed.
            })  # Finish adding the validation feedback.

        except Exception as e:  # Catch another ordinary error and store it in e.
            results.append({  # Add an error reply to results.
                "type": "tool_result",  # Identify this entry as a tool result.
                "tool_use_id": block.id,  # Match the error to the original request.
                "content": f"Error: {e}",  # Include the error message.
                "is_error": True,  # Indicate that the request failed.
            })  # Finish adding the error result.

    return results, report  # Return two values: tool replies and the validated report, or None.

In [109]:
# 2. Investigate the incident until Claude submits a valid structured report.

def triage(incident):  # incident contains the user's description of the problem.
    messages = [{"role": "user", "content": incident}]  # Start the conversation with the incident description.

    for turn in range(MAX_TURNS):  # Repeat up to MAX_TURNS times; turn is the current loop number.
        response = chat(messages)  # Ask Claude for its next response.
        messages.append({  # Save Claude's complete reply in the conversation.
            "role": "assistant",  # Identify the reply as Claude's.
            "content": response.content,  # Preserve all text and tool-request blocks.
        })  # Finish saving the reply.

        if response.stop_reason != "tool_use":  # Check whether Claude replied without requesting tools.
            messages.append({  # Add a reminder to submit the report through the tool.
                "role": "user",  # Send the reminder using the user role.
                "content": "Please submit your findings with submit_triage_report.",  # Request the structured submission.
            })  # Finish adding the reminder.
            continue  # Start the next loop iteration without running tools.

        tool_results, report = run_tools(response)  # Unpack the two returned values into tool_results and report.

        if report is not None:  # Check whether a report passed validation.
            return report  # Return the validated report and exit triage().

        messages.append({  # Add tool results or validation errors for Claude's next attempt.
            "role": "user",  # Tool results are sent using the user role.
            "content": tool_results,  # Supply the collected result blocks.
        })  # Finish adding the tool feedback.

    raise RuntimeError(f"No valid report after {MAX_TURNS} turns")  # Stop with an error if no valid report was returned.

In [110]:
# 3. Run the investigation and display the validated report as formatted JSON.

report = triage("Customers say checkout is failing with 500 errors since about 2pm.")  # Investigate the incident and store the validated report.
print(report.model_dump_json(indent=2))  # Convert report into JSON text with two-space indentation and print it.

-> Claude called list_services({})
-> Claude called get_service_metrics({'service_name': 'checkout-service'})
-> Claude called get_service_metrics({'service_name': 'payments-api'})
-> Claude called get_service_metrics({'service_name': 'auth-service'})
-> Claude called submit_triage_report({'severity': 'SEV1', 'summary': 'Checkout service is experiencing severe degradation with 7.4% error rate and 2300ms p95 latency since ~2pm, preventing customers from completing purchases. A deploy of v2.18.0 introducing new promo-code discount logic occurred at 1:52pm, immediately prior to customer reports.', 'affected_services': ['checkout-service'], 'likely_cause': 'The v2.18.0 deploy at 13:52 introducing new promo-code discount logic has a bug causing checkout requests to fail. The timing correlation is strong: deploy at 1:52pm, customer reports starting ~2pm.', 'evidence': ['checkout-service error rate is 7.4% (threshold for SEV1 is 5%)', 'checkout-service p95 latency is 2300ms (threshold for SEV

## **Model Evaluation**

**This evaluation tests your triage assistant on four incidents.** Each report receives three code-based checks:

| Check | Passes when… |
|---|---|
| **Severity** (`severity_ok`) | The report's severity is one of the **allowed** values for that incident |
| **Affected services** (`services_ok`) | The report names **every expected** service |
| **No invented versions** (`no_invented_versions`) | Every version number mentioned (like `v2.18.0`) **really exists** in `SERVICES` — this catches made-up ones like `v2.17.x` |

**Run the cells in order:** 1 → 2 → 3 → 4. Cell 4 calls the API (one full triage per incident).

**Two Python tools used below:**

- `.items()` gives each key and value from a dictionary:

```python
person = {"name": "Mohammad"}
for key, value in person.items():
    print(key, value)  # name Mohammad
```

- `enumerate()` gives each item and its position while looping:

```python
for position, fruit in enumerate(["apple", "banana"], start=1):
    print(position, fruit)  # 1 apple, then 2 banana
```

### **1. The test dataset**

Cases 1–2 are real checkout problems; cases 3–4 are minor reports about healthy services, so a good assistant should say **SEV3**.

In [112]:
# 1. Create a dataset of incidents and their expected results.

EVAL_CASES = [  # EVAL_CASES stores a list of test-case dictionaries.
    {  # Start the first test case.
        "incident": "Customers say checkout is failing with 500 errors since about 2pm.",  # Describe the problem.
        "allowed_severity": ["SEV1", "SEV2"],  # Either severity is acceptable for this test.
        "expected_services": ["checkout-service"],  # The report should include this service.
    },  # Finish the first case.
    {  # Start the second test case.
        "incident": "The checkout page has become very slow for everyone this afternoon.",  # Describe a slow checkout.
        "allowed_severity": ["SEV1", "SEV2"],  # Accept either of these severity levels.
        "expected_services": ["checkout-service"],  # Expect checkout to be included.
    },  # Finish the second case.
    {  # Start the third test case.
        "incident": "One customer said their card payment was declined once, but it worked on retry.",  # Describe a temporary payment issue.
        "allowed_severity": ["SEV3"],  # Only SEV3 passes this test.
        "expected_services": ["payments-api"],  # Expect the payments service to be included.
    },  # Finish the third case.
    {  # Start the fourth test case.
        "incident": "A couple of users mentioned login felt slightly slow this morning.",  # Describe a minor login complaint.
        "allowed_severity": ["SEV3"],  # Only SEV3 passes this test.
        "expected_services": ["auth-service"],  # Expect the authentication service to be included.
    },  # Finish the fourth case.
]  # Finish the dataset.

print(len(EVAL_CASES))  # Print the number of test cases: 4.
print(EVAL_CASES[2]["allowed_severity"])  # Print the third case's allowed severities: ['SEV3']; list positions start at zero.

4
['SEV3']


### **2. The grading function**

`grade(case, report)` compares one report with its test case and returns the three checks plus a score such as `"2/3"`.

In [ ]:
# 2. Collect the known software versions and define the grading function.

import re  # Import Python's regular-expression tools for finding text patterns.

REAL_VERSIONS = {  # REAL_VERSIONS is a set containing unique version names from SERVICES.
    d["version"]  # Extract the version name from deployment dictionary d.
    for s in SERVICES.values()  # s holds each service's details, without its dictionary key.
    for d in s["recent_deploys"]  # d holds each recorded deployment for that service.
}  # Finish collecting the known versions.


def grade(case, report):  # case contains expected results; report is the validated TriageReport to check.
    text = report.model_dump_json()  # Convert the report into JSON text so its contents can be searched.
    mentioned = re.findall(r"v\d+\.\d+(?:\.\d+|\.x)?", text)  # Store matching version strings found anywhere in text.

    checks = {  # checks stores a True/False result for each grading rule.
        "severity_ok": report.severity in case["allowed_severity"],  # Pass if the report's severity is allowed.
        "services_ok": all(  # Pass only if every expected service appears in the report.
            s in report.affected_services  # Check whether service name s appears in the reported list.
            for s in case["expected_services"]  # Examine each expected service name.
        ),  # Finish the service check.
        "no_invented_versions": all(  # Pass only if every matched version exists in the known set.
            v in REAL_VERSIONS  # Check whether version string v is recorded in the sample data.
            for v in mentioned  # Examine each version found in the report.
        ),  # Finish the version check.
    }  # Finish the three checks.

    checks["score"] = f"{sum(checks.values())}/3"  # Count True values as 1 and False values as 0; store a score such as "2/3".
    return checks  # Return the check results and score.

**The regular expression, briefly:**

| Part | Meaning |
|---|---|
| `r"..."` | A raw string that preserves backslashes for the pattern. |
| `v` | Match the letter `v`. |
| `\d+` | Match one or more digits. |
| `\.` | Match an actual dot. |
| `(?:...)` | Group part of the pattern without returning it separately. |
| `\.\d+` or `\.x` | Match a dot followed by digits, or `.x`. |
| `?` | Make the preceding group optional. |

For example, it finds `v2.18.0`, `v2.17.x` and `v2.18`.

### **3. Check the grader catches an invented version (no API call)**

**Why does `fake` pass Pydantic validation?** Its fields satisfy the model's rules. Your separate grader is what checks whether its version references exist in the sample data. (`good` is the example report from the Pydantic step earlier.)

In [114]:
# 3. Check that the grader detects an unrecorded software version.

print(REAL_VERSIONS)  # Print the known versions; sets do not guarantee display order.
# Example output: {'v2.18.0', 'v5.2.1'}

fake = TriageReport.model_validate({  # fake stores a structurally valid report with a deliberately unsupported version.
    **good,  # Copy the fields from your earlier good example.
    "next_steps": ["Roll back to v2.17.x"],  # Replace next_steps with an action mentioning an unrecorded version.
})  # Validate the structure and create the TriageReport object.

print(grade(EVAL_CASES[0], fake))  # Grade fake against the first test case and display the results.
# Expected: {'severity_ok': True, 'services_ok': True, 'no_invented_versions': False, 'score': '2/3'}

{'v2.18.0', 'v5.2.1'}
{'severity_ok': True, 'services_ok': True, 'no_invented_versions': False, 'score': '2/3'}


### **4. Run the full evaluation**

Each case runs a real `triage()`, gets graded, and is printed. If one case crashes, it is recorded as failed and the eval carries on. **Run it at least twice** — results can differ between runs.

In [116]:
# 4. Run every test case and calculate the overall percentage of checks passed.

def run_eval():  # Define a function that evaluates all cases in EVAL_CASES.
    passed, total = 0, 0  # passed counts successful checks; total counts all checks, including failed cases.

    for i, case in enumerate(EVAL_CASES, 1):  # i is the case number starting at 1; case is its dictionary.
        try:  # Attempt to investigate and grade this incident.
            report = triage(case["incident"])  # Run the assistant and store its validated report.
            checks = grade(case, report)  # Compare the report with the case's expected results.

            passed += sum(  # Add this case's successful checks to passed.
                v  # Use each check's Boolean value: True counts as 1, False as 0.
                for k, v in checks.items()  # k is each dictionary key; v is its value.
                if k != "score"  # Exclude the score string, such as "3/3".
            )  # Finish counting successful checks.

            print(f"Case {i}: {report.severity} {report.affected_services} -> {checks}")  # Display the case number, report details and grading results.

        except Exception as e:  # Catch a failure and store the error in e.
            print(f"Case {i}: FAILED ({e})")  # Display the failure; the loop can continue to other cases.

        total += 3  # Count three checks for this case, even if it failed.

    print(f"\nOverall: {passed}/{total} checks passed ({passed / total:.0%})")  # Print the totals and percentage, rounded to zero decimal places.


run_eval()  # Execute the evaluation; triage() makes API requests for each incident.

-> Claude called list_services({})
-> Claude called get_service_metrics({'service_name': 'checkout-service'})
-> Claude called submit_triage_report({'severity': 'SEV1', 'summary': 'Checkout-service is experiencing 7.4% error rate and 2300ms p95 latency following a deployment at 13:52 that introduced new promo-code discount logic. Customers cannot complete purchases.', 'affected_services': ['checkout-service'], 'likely_cause': 'The v2.18.0 deployment at 13:52 introducing new promo-code discount logic has caused checkout failures with high error rates and elevated latency.', 'evidence': ['checkout-service error rate: 7.4% (critical threshold is 5%)', 'checkout-service p95 latency: 2300ms (critical threshold is 2000ms)', "v2.18.0 deployed at 13:52 with change: 'New promo-code discount logic'", 'Customer reports of 500 errors in checkout since ~2pm, correlating with deployment time'], 'next_steps': ['Immediately rollback checkout-service to the previous stable version', 'Monitor checkout-s

**How to interpret the score:** four cases × three checks gives **12 checks**. If ten pass, the output is:

```text
Overall: 10/12 checks passed (83%)
```

**Limits of these checks:**

- `services_ok` allows extra services; it only checks that the expected ones are included.
- The version check passes if no versions are mentioned. It also does not verify that a known version belongs to the correct service.
- Cases 3–4 describe potentially minor issues, but the code does not prove they are false alarms. All cases use the same `SERVICES` snapshot, which still contains the checkout problem.
- The score measures these three rules — not overall diagnostic accuracy. It also doesn't check the *process* (e.g. whether Claude looked at every service).